# **About Gradient Boosting**

Source: [Homemade Machine Learning repository](https://github.com/trekhleb/homemade-machine-learning)

Before moving on with this demo you might want to take a look at:

* [Ensemble Learning – Wikipedia](https://en.wikipedia.org/wiki/Ensemble_learning)
* [Gradient Boosting – Wikipedia](https://en.wikipedia.org/wiki/Gradient_boosting)
* [Greedy Function Approximation: A Gradient Boosting Machine (Friedman 2001)](https://statweb.stanford.edu/~jhf/ftp/trebst.pdf)

Gradient Boosting is a powerful ensemble learning method that builds a strong predictive model by combining many weak learners (typically decision trees) in a sequential manner. Unlike Random Forest which builds trees in parallel, Gradient Boosting builds trees one after another, where each new tree tries to correct the errors made by the previous ones.

## Math Recap

The ensemble is an **additive model**:

$$F_M(x) = F_0 + \sum_{m=1}^{M} \eta \cdot h_m(x)$$

We minimise **log-loss** (binary cross-entropy):

$$L(y, p) = -\frac{1}{n}\sum_{i=1}^{n}\left[y_i \log p_i + (1-y_i)\log(1-p_i)\right]$$

At each round we fit a tree to the **pseudo-residuals** (negative gradient of the loss):

$$r_i = y_i - \sigma\!\left(F_{m-1}(x_i)\right)$$

where $\sigma$ is the sigmoid function.

## Import Dependencies

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Homemade implementation — no sklearn ensemble here.
from homemade.gradient_boosting import GradientBoostingClassifier

## Load the Data

In this demo we will use a server operational parameters dataset where **Latency (ms)** and **Throughput (mb/s)** help us detect anomalous server behaviour (`Anomaly = 1`). Normal behaviour is marked as `0`.

In [ ]:
data = pd.read_csv(
    'https://raw.githubusercontent.com/trekhleb/homemade-machine-learning'
    '/master/data/server-operational-params.csv'
)
data.head(10)

## Explore the Dataset

In [ ]:
print(f"Dataset shape: {data.shape}")
print(f"\nColumn names: {data.columns.tolist()}")
print(f"\nClass distribution:")
print(data['Anomaly'].value_counts())
print(f"\nBasic statistics:")
data.describe()

## Plot the Data

Visualising the data helps us understand the relationship between latency, throughput, and anomalies.

In [ ]:
classes = [0, 1]
x_axis = 'Latency (ms)'
y_axis = 'Throughput (mb/s)'

for anomaly_class in classes:
    plt.scatter(
        data[x_axis][data['Anomaly'] == anomaly_class],
        data[y_axis][data['Anomaly'] == anomaly_class],
        label=f'Anomaly={anomaly_class}',
        alpha=0.7
    )

plt.xlabel(x_axis)
plt.ylabel(y_axis)
plt.title('Server Operational Parameters')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Prepare the Data for Training

In [ ]:
X_train = data[['Latency (ms)', 'Throughput (mb/s)']].values
y_train = data['Anomaly'].values

## Train the Homemade Gradient Boosting Model

In [ ]:
n_estimators = 100        # Number of boosting stages.
learning_rate = 0.1       # Step size shrinkage.
max_depth = 3             # Maximum depth of each tree.
min_samples_split = 2     # Minimum samples to split a node.

gradient_boosting = GradientBoostingClassifier(
    n_estimators=n_estimators,
    learning_rate=learning_rate,
    max_depth=max_depth,
    min_samples_split=min_samples_split,
)

gradient_boosting.fit(X_train, y_train)
print("Training complete.")

## Loss Curve

Gradient Boosting minimises log-loss (binary cross-entropy). Our homemade classifier records the loss after every boosting round in `train_loss_`. A steadily decreasing curve means the model is learning.

In [ ]:
losses = gradient_boosting.train_loss_

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(losses) + 1), losses)
plt.xlabel('Boosting Round')
plt.ylabel('Log-Loss')
plt.title('Training Loss over Boosting Rounds (Homemade)')
plt.grid(True, alpha=0.3)
plt.show()

print(f"Initial loss : {losses[0]:.4f}")
print(f"Final loss   : {losses[-1]:.4f}")

## Model Evaluation

In [ ]:
y_pred = gradient_boosting.predict(X_train)

# Compute metrics without sklearn.
accuracy  = np.mean(y_pred == y_train)

true_pos  = np.sum((y_pred == 1) & (y_train == 1))
false_pos = np.sum((y_pred == 1) & (y_train == 0))
false_neg = np.sum((y_pred == 0) & (y_train == 1))
true_neg  = np.sum((y_pred == 0) & (y_train == 0))

precision = true_pos / (true_pos + false_pos) if (true_pos + false_pos) > 0 else 0.0
recall    = true_pos / (true_pos + false_neg) if (true_pos + false_neg) > 0 else 0.0
f1        = (2 * precision * recall / (precision + recall)
             if (precision + recall) > 0 else 0.0)

print('Model Performance Metrics (Homemade):')
print('=' * 35)
print(f'Accuracy  : {accuracy:.4f} ({accuracy*100:.2f}%)')
print(f'Precision : {precision:.4f}')
print(f'Recall    : {recall:.4f}')
print(f'F1-Score  : {f1:.4f}')
print('\nConfusion Matrix:')
print(f'  TP={true_pos}  FP={false_pos}')
print(f'  FN={false_neg}  TN={true_neg}')

## Decision Boundary

Because we have only two features we can plot the entire 2-D feature space coloured by the model's prediction.

- **Blue** points = Normal (label 0)
- **Red** points = Anomaly (label 1)
- **Background colour** = predicted class

The boundary is non-linear because gradient boosting uses decision trees.

In [ ]:
x_min, x_max = X_train[:, 0].min() - 0.5, X_train[:, 0].max() + 0.5
y_min, y_max = X_train[:, 1].min() - 0.5, X_train[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                     np.linspace(y_min, y_max, 200))

Z = gradient_boosting.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

plt.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.RdYlBu)
plt.scatter(X_train[y_train == 0, 0], X_train[y_train == 0, 1],
            label='Normal', c='blue', edgecolors='k')
plt.scatter(X_train[y_train == 1, 0], X_train[y_train == 1, 1],
            label='Anomaly', c='red', edgecolors='k')
plt.xlabel('Latency (ms)')
plt.ylabel('Throughput (mb/s)')
plt.title('Decision Boundary (Homemade Gradient Boosting)')
plt.legend()
plt.show()

## Anomaly Probability Heatmap

We can also visualise the model's confidence — the predicted probability of anomaly — across the feature space.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

Z_proba = gradient_boosting.predict_proba(
    np.c_[xx.ravel(), yy.ravel()]
)[:, 1].reshape(xx.shape)

contour = ax.contourf(xx, yy, Z_proba, levels=20,
                      cmap=plt.cm.RdYlBu, alpha=0.8)
ax.scatter(X_train[y_train == 0, 0], X_train[y_train == 0, 1],
           label='Normal (0)', c='blue', edgecolors='k', alpha=0.5, s=30)
ax.scatter(X_train[y_train == 1, 0], X_train[y_train == 1, 1],
           label='Anomaly (1)', c='red', edgecolors='k', alpha=0.5, s=30)
ax.set_xlabel('Latency (ms)')
ax.set_ylabel('Throughput (mb/s)')
ax.set_title('Anomaly Probability Heatmap (Homemade)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.colorbar(contour, ax=ax, label='P(Anomaly)')
plt.tight_layout()
plt.show()

This demo shows the core ideas of gradient boosting implemented from scratch:

- **Sequential error correction** — each tree targets the residuals of the current ensemble.
- **Decreasing log-loss** — the training curve confirms learning.
- **Flexible non-linear boundaries** — trees can carve out complex regions in feature space.

Try changing `n_estimators`, `learning_rate`, and `max_depth` to see how each hyperparameter affects the decision boundary and accuracy.